# Emotion Music Studio — free GPU backend (Colab)

**⚠️ Run this ON Google Colab, not on your own computer.** It needs Colab's Linux GPU
runtime + tools (`cloudflared`, `chmod`). Running it locally on Windows fails with
`FileNotFoundError [WinError 2]`.

Open it in Colab:
`https://colab.research.google.com/github/AthSri0507/Multi_Modal-Music-Generation/blob/main/deploy/colab_gpu_backend.ipynb`

Then: **Runtime → Change runtime type → T4 GPU**, and run the cells top to bottom.

(To run on your own Windows PC instead — CPU, no GPU — you don't need this notebook at
all: just `uvicorn src.api.app:app --port 8000` and open http://localhost:8000.)

In [ ]:
# 0. Guard: make sure we're actually on Colab (Linux). Stops the WinError 2 confusion.
import os, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not (IN_COLAB and os.name == 'posix'):
    raise SystemExit(
        'STOP: open this notebook in Google Colab (colab.research.google.com), not locally.\n'
        'To run locally on Windows instead, skip this notebook and run:\n'
        '    uvicorn src.api.app:app --port 8000   (then open http://localhost:8000)')
print('On Colab — good to go.')

In [ ]:
# 1. Clone your repo
REPO_URL = "https://github.com/AthSri0507/Multi_Modal-Music-Generation.git"
if not os.path.exists('repo'):
    !git clone --depth 1 $REPO_URL repo
%cd repo

In [ ]:
# 2. Install serving deps. Colab already ships a CUDA build of torch, so do NOT
#    reinstall torch (that would downgrade it / break CUDA).
!pip install -q transformers sentencepiece protobuf soundfile fastapi uvicorn python-multipart librosa
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to T4 GPU')

In [ ]:
# 3. Sanity-check the app imports BEFORE launching the server (surfaces any error here,
#    not as a silent 502 later).
os.environ['PYTHONPATH'] = os.getcwd()
import importlib, src.api.app as _app  # noqa
print('app imports OK')

In [ ]:
# 4. (optional) build the React UI so the backend serves it at /
import shutil
if shutil.which('node'):
    !cd frontend && npm install --silent && npm run build
else:
    print('node not found; API-only.')

In [ ]:
# 5. Launch uvicorn (robustly) and WAIT until it is actually healthy.
import subprocess, time, urllib.request
env = {**os.environ, 'PYTHONPATH': os.getcwd(),
       'MUSICGEN_MODEL_NAME': 'facebook/musicgen-medium',  # GPU handles this easily
       'MUSICGEN_USE_CLAP': '1'}                            # Stage-2 CLAP ranking
log = open('server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'src.api.app:app',
                           '--host', '0.0.0.0', '--port', '8000'],
                          cwd=os.getcwd(), env=env, stdout=log, stderr=subprocess.STDOUT)
ok = False
for _ in range(60):  # up to ~2 min
    try:
        r = urllib.request.urlopen('http://localhost:8000/api/v1/health', timeout=3)
        print('✅ BACKEND UP:', r.read().decode()); ok = True; break
    except Exception:
        time.sleep(2)
if not ok:
    print('❌ backend did not start — error log:\n')
    print(open('server.log').read()[-3000:])

In [ ]:
# 6. (recommended) Pre-download + warm the model so the first PUBLIC request isn't a
#    multi-minute block that Cloudflare times out into a 502. Downloads ~3.5GB once.
import urllib.request, json
print('warming up musicgen-medium (downloads ~3.5GB on first run, then generates once)...')
req = urllib.request.Request('http://localhost:8000/api/v1/music/generate',
    data=json.dumps({'prompt': 'warm up', 'duration': 3, 'preset': 'fast'}).encode(),
    headers={'Content-Type': 'application/json'})
print('warmup done:', json.loads(urllib.request.urlopen(req, timeout=900).read()).get('id'))

In [ ]:
# 7. Public URL via cloudflared (Linux binary — Colab only)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    print(line.strip())
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n==== PUBLIC URL:', url, '====\nOpen it in a browser, or use it as VITE_API_BASE for the local React app.')

In [ ]:
# 8. Test the public URL (handles non-200 without crashing).
import requests
h = requests.get(url + '/api/v1/health')
print('health', h.status_code, h.text[:200])
r = requests.post(url + '/api/v1/music/generate',
                  json={'prompt': 'happy upbeat jazz trio', 'duration': 10, 'preset': 'balanced'})
print('generate', r.status_code, (r.json() if r.headers.get('content-type','').startswith('application/json') else r.text[:200]))

**Troubleshooting**
- 502 / `JSONDecodeError` from the URL ⇒ the backend isn't responding. Re-run cell 5
  (it prints `server.log` if startup failed) and cell 6 (warm-up) before tunnelling.
- If `device` is `cpu`, set Runtime → Change runtime type → **T4 GPU** and re-run.
- The gallery (SQLite) + audio reset when the runtime stops; mount Drive to persist.
- The free tunnel URL changes each session.